# Módulo 47: Medidas y KPIs — Guía del ejercicio

En este ejercicio vas a escribir **medidas** y **columnas calculadas** sobre el modelo de ventas con DuckDB y pandas: totales y promedios, comparaciones con el mes y el año anteriores, acumulado del año y diferencia mensual. Después validarás los resultados en una tabla por país y mes y en un gráfico.

- Si necesitas repasar los conceptos, consulta la lección [m47-medidas-y-kpis-leccion.ipynb](m47-medidas-y-kpis-leccion.ipynb).
- Cuando termines, compara tu trabajo con la solución: [m47-medidas-y-kpis-solucion.ipynb](m47-medidas-y-kpis-solucion.ipynb).

## Contexto

El tablero de ventas ya muestra totales, pero la dirección comercial empieza a hacer preguntas que una suma no responde: "¿vendimos más que el mes pasado?", "¿y que el mismo mes del año anterior?", "¿cuánto llevamos acumulado en el año?". También quiere conocer el importe de las ventas con impuestos y el precio con un descuento promocional.

Para responder, el equipo de análisis necesita un conjunto de medidas ordenadas en un solo lugar, escritas como funciones que reciben el contexto de filtro (países y fechas), y una tabla de validación donde comprobar, país por país y mes a mes, que los cálculos son correctos.

## Datos

Trabajas sobre el modelo estrella del Módulo 45, que [`../practica-ventas/ventas.py`](../practica-ventas/ventas.py) carga en DuckDB con `practica.conectar()`:

| Tabla o vista | Filas | Columnas que usarás |
| --- | --- | --- |
| `ventas` | 1 499 000 (del 1 de enero de 2019 al 31 de diciembre de 2021) | `fecha`, `cantidad`, `precio`, `importe` (= `cantidad` × `precio`) |
| `ventas_detalle` | Las mismas ventas, unidas a sus dimensiones | `pais`, `categoria`, `anio`, `trimestre`, `mes`, `anio_mes` |
| `calendario` | 1 096 (un día por fila, 2019–2021) | `fecha`, `anio`, `trimestre`, `mes`, `nombre_mes`, `anio_mes` |
| `paises` | 10 | `id_pais`, `pais` |

`practica.filtro_sql(paises=..., desde=..., hasta=...)` traduce un contexto de filtro a una cláusula `WHERE` con parámetros, y `practica.ventas_mensuales(con, por="pais")` calcula las medidas de tiempo de referencia: úsala solo para **comprobar** tus resultados.

## Instrucciones

- Trabaja en este notebook, ejecutándolo desde su carpeta para que las rutas relativas funcionen (`uv run jupyter lab` desde la raíz del repositorio).
- Escribe las medidas con SQL sobre la conexión `con` (DuckDB) o con pandas sobre resultados ya agregados. No cargues los 1,5 millones de filas en pandas si una consulta agregada basta.
- Cada medida es una función que recibe `con` y el contexto de filtro como argumentos (`paises`, `desde`, `hasta`), y devuelve `None` (el "blanco") cuando el contexto no tiene ventas.
- Después de cada tarea, comprueba el resultado con un par de valores conocidos y explica en una frase qué hace la medida.

La celda siguiente importa las librerías y abre el modelo.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "png"
pio.defaults.default_width = 900
pio.defaults.default_height = 450

sys.path.insert(0, str(Path("../practica-ventas").resolve()))
import ventas as practica

con = practica.conectar()

## Tarea 1. Un módulo de medidas

1. Escribe una función auxiliar que reciba `con`, una expresión agregada en SQL (por ejemplo, `"sum(importe)"`) y el contexto de filtro, aplique `practica.filtro_sql` sobre `ventas_detalle` y devuelva un solo valor.
2. Crea un diccionario `MEDIDAS` y un decorador `registrar` que agregue a ese diccionario cada función que decore. Así todas las medidas de las tareas siguientes quedan reunidas en un solo lugar, aunque se definan en celdas distintas.
3. (Opcional) Si prefieres un archivo, reúnelas en un `medidas.py` junto a este notebook e impórtalo.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 2. Medidas básicas

1. `ventas_total(con, **filtros)`: la suma de `importe`.
2. `ventas_promedio(con, **filtros)`: el importe promedio por venta, el promedio de `importe`.
3. Comprueba que `ventas_total(con)` da 41 940 958 825 y que, para Canadá en 2020, da 856 197 372.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 3. Comparaciones con el mes y el año anteriores

1. `ventas_mes_anterior(con, anio, mes, **filtros)`: el valor de `ventas_total` en el **mes anterior** al mes indicado.
2. `ventas_mismo_mes_anio_anterior(con, anio, mes, **filtros)`: el valor de `ventas_total` en el **mismo mes del año anterior**.

Ambas deben reutilizar `ventas_total` y cambiar solo el rango de fechas del contexto. Comprueba que en enero de 2019 las dos devuelven `None`, porque no hay datos previos.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 4. Acumulado del año y diferencia mensual

1. `ventas_ytd(con, anio, mes, **filtros)`: las ventas acumuladas desde el 1 de enero del año hasta el último día del mes indicado.
2. `dif_mensual(con, anio, mes, **filtros)`: la diferencia entre las ventas del mes y las del mes anterior,

$$\text{dif\_mensual} = \text{ventas del mes} - \text{ventas\_mes\_anterior}.$$

Si no hay mes anterior, la diferencia también debe quedar en blanco.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 5. Columnas calculadas

Crea en DuckDB una vista `ventas_calculadas` sobre `ventas_detalle` con dos columnas calculadas:

1. `ventas_con_impuestos`: el importe de cada venta con un impuesto del 20 %.
2. `precio_con_descuento`: el precio unitario con un descuento del 10 %.

A diferencia de las medidas, estas columnas se calculan fila a fila, así que se pueden sumar, filtrar o agrupar en cualquier consulta. Comprueba que la suma de `ventas_con_impuestos` es exactamente 1.2 veces las ventas totales.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Tabla de validación

1. Construye una **tabla** con `pais`, `anio` y `mes`, y las medidas `ventas`, `ventas_mes_anterior`, `ventas_mismo_mes_anio_anterior`, `ventas_ytd` y `dif_mensual`, para todos los países y los 36 meses del calendario. Hazlo con una sola consulta (funciones de ventana sobre la rejilla de meses × países), no llamando 1 800 veces a las funciones de las tareas 3 y 4.
2. Revisa la tabla: el "mes anterior" de una fila debe coincidir con `ventas` de la fila de arriba (dentro del mismo país), el acumulado debe reiniciarse en enero y algunas celdas deben coincidir con las funciones de las tareas anteriores. Compárala también con `practica.ventas_mensuales(con, por="pais")`.
3. Muestra la tabla con un formato numérico legible y guárdala como CSV.
4. Dibuja un **gráfico de barras agrupadas** por país que compare la suma de `importe` con la suma de `ventas_con_impuestos`.
5. (Opcional) Lleva la tabla, el gráfico y un mapa de burbujas por país (tamaño según `ventas_promedio`, con `ventas_total` en la información emergente) a una app de Streamlit con filtro de países y un botón para descargar el CSV. Compárala con [`m47-medidas-y-kpis-app.py`](m47-medidas-y-kpis-app.py).

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] Las seis medidas están registradas en `MEDIDAS` y aceptan el contexto de filtro como argumentos.
- [ ] Las comparaciones con el mes y el año anteriores dejan en blanco los periodos sin datos previos (por ejemplo, enero de 2019).
- [ ] La vista `ventas_calculadas` tiene las dos columnas calculadas.
- [ ] La tabla de validación tiene 360 filas (10 países × 36 meses), coincide con `practica.ventas_mensuales` y está guardada como CSV.
- [ ] El gráfico de barras agrupadas compara las ventas con y sin impuestos por país.

Cuando termines, compara tu trabajo con la solución: [m47-medidas-y-kpis-solucion.ipynb](m47-medidas-y-kpis-solucion.ipynb).